In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_DIR = PROJECT_ROOT / "results"

In [4]:
regime_features = pd.read_csv(
    RESULTS_DIR / "regime_features_2025.csv",
    index_col=0,
    parse_dates=True
)

monthly_analysis = pd.read_csv(
    RESULTS_DIR / "monthly_regime_strategy_analysis_2025.csv",
    index_col=0
)

eiie_concentration_summary = pd.read_csv(
    RESULTS_DIR / "eiie_concentration_summary.csv"
)

display(regime_features.head())

display(monthly_analysis)

display(eiie_concentration_summary)

,Breadth_20D,Momentum_20D,Dispersion_20D
date,,,
2025-04-16,0.45,-0.088275,0.019067
2025-04-17,0.45,-0.096330,0.018548
2025-04-18,0.45,-0.092261,0.018449
2025-04-21,0.47,-0.088425,0.017408
2025-04-22,0.46,-0.088465,0.017421


,Breadth_20D,Momentum_20D,Dispersion_20D,Static MVO,Rolling MVO,Equal Weight,EIIE 40ep,NAVER Only,Winner,Winner Return (%)
2025-04,0.471818,-0.058412,0.017376,-4.255535,-0.022066,0.925554,4.250478,4.973826,NAVER Only,4.973826
2025-05,0.523684,0.073211,0.015165,2.852853,13.954122,4.838804,-5.824894,-6.483807,Rolling MVO,13.954122
2025-06,0.572105,0.152187,0.018870,12.350584,6.847919,20.800284,37.054724,39.999989,NAVER Only,39.999989
2025-07,0.533043,0.085824,0.024159,12.978674,-6.063922,1.399298,-7.996439,-10.476197,Static MVO,12.978674
2025-08,0.503000,-0.011723,0.018423,-1.066469,-4.945377,-1.912635,-7.417444,-8.723370,Static MVO,-1.066469
2025-09,0.593810,0.081579,0.016604,17.973397,21.121668,15.707883,22.679824,25.174811,NAVER Only,25.174811
2025-10,0.573333,0.193840,0.024320,33.939450,23.440032,23.405333,-0.099516,-0.372445,Static MVO,33.939450
2025-11,0.529500,0.109388,0.024206,-6.574137,-7.175859,-4.092022,-7.395266,-8.785043,Equal Weight,-4.092022
2025-12,0.466667,0.027111,0.020100,19.399951,19.059271,10.946745,-0.440232,-0.614773,Static MVO,19.399951


,Metric,Value
0,EIIE_NAVER_Correlation,0.998752
1,Average_HHI,0.772454
2,Effective_Number_of_Assets,1.300079
3,Average_Max_Weight,0.871604


In [5]:
stock_prices_2025 = pd.read_csv(
    RESULTS_DIR / "stock_prices_2025.csv",
    index_col=0,
    parse_dates=True
)

stock_prices_2025.head()

,000660.KS,005380.KS,005930.KS,035420.KS,105560.KS
date,,,,,
2025-03-19,204231.625000,194257.859375,57083.609375,205837.796875,77372.546875
2025-03-20,208703.828125,192829.468750,58742.453125,205837.796875,78420.695312
2025-03-21,214169.875000,195210.109375,60206.136719,207322.203125,77467.835938
2025-03-24,210194.578125,202828.062500,59035.191406,204848.187500,77944.265625
2025-03-25,206716.187500,209493.765625,58352.140625,205342.984375,78039.562500


In [6]:
#종목별 20일 momentum
#20거래일 누적수익률

stock_momentum_20d = (
    stock_prices_2025
    / stock_prices_2025.shift(20)
    - 1
)

stock_momentum_20d.head(25)


,000660.KS,005380.KS,005930.KS,035420.KS,105560.KS
date,,,,,
2025-03-19,NaN,NaN,NaN,NaN,NaN
2025-03-20,NaN,NaN,NaN,NaN,NaN
2025-03-21,NaN,NaN,NaN,NaN,NaN
2025-03-24,NaN,NaN,NaN,NaN,NaN
2025-03-25,NaN,NaN,NaN,NaN,NaN
2025-03-26,NaN,NaN,NaN,NaN,NaN
2025-03-27,NaN,NaN,NaN,NaN,NaN
2025-03-28,NaN,NaN,NaN,NaN,NaN
2025-03-31,NaN,NaN,NaN,NaN,NaN


In [7]:
stock_momentum_20d.loc["2025-10-01"]

000660.KS    0.381958
005380.KS   -0.020455
005930.KS    0.250126
035420.KS    0.131111
105560.KS    0.068934
Name: 2025-10-01 00:00:00, dtype: float64

In [9]:
# 20일 Momentum이 계산된 날짜만 사용
valid_stock_momentum_20d = (
    stock_momentum_20d
    .dropna(how="all")
)

# 가장 강한 종목
leader_ticker = (
    valid_stock_momentum_20d
    .idxmax(axis=1)
)

# 가장 높은 20일 Momentum
leader_return = (
    valid_stock_momentum_20d
    .max(axis=1)
)

leadership = pd.DataFrame({
    "Leader": leader_ticker,
    "Leader_Momentum_20D": leader_return
})

leadership.head()

,Leader,Leader_Momentum_20D
date,,
2025-04-16,105560.KS,-0.014778
2025-04-17,105560.KS,-0.024301
2025-04-18,105560.KS,0.013530
2025-04-21,105560.KS,0.012225
2025-04-22,105560.KS,0.018315


In [10]:
#leadership gap

sorted_momentum = np.sort(
    valid_stock_momentum_20d.values,
    axis=1
)

leadership_gap = pd.Series(
    sorted_momentum[:, -1]
    - sorted_momentum[:, -2],
    index=valid_stock_momentum_20d.index,
    name="Leadership_Gap_20D"
)

leadership_gap.head()

date
2025-04-16    0.044624
2025-04-17    0.054978
2025-04-18    0.100359
2025-04-21    0.091082
2025-04-22    0.084821
Name: Leadership_Gap_20D, dtype: float64

In [16]:
#semiconductor leadership

semiconductor_momentum_20d = (
    stock_momentum_20d[
        [
            "000660.KS",
            "005930.KS"
        ]
    ]
    .mean(axis=1)
)

naver_momentum_20d = (
    stock_momentum_20d[
        "035420.KS"
    ]
)

auto_momentum_20d = (
    stock_momentum_20d[
        "005380.KS"
    ]
)

finance_momentum_20d = (
    stock_momentum_20d[
        "105560.KS"
    ]
)

In [17]:
#momemtum 변수 기준 통일

semiconductor_momentum_20d = (
    valid_stock_momentum_20d[
        ["000660.KS", "005930.KS"]
    ]
    .mean(axis=1)
)

naver_momentum_20d = (
    valid_stock_momentum_20d["035420.KS"]
)

auto_momentum_20d = (
    valid_stock_momentum_20d["005380.KS"]
)

finance_momentum_20d = (
    valid_stock_momentum_20d["105560.KS"]
)

universe_momentum_20d = (
    valid_stock_momentum_20d
    .mean(axis=1)
)

In [18]:
#상대강도
#Relative Strength = 종목 Momentum - 전체 Universe Momentum

naver_relative_strength = (
    naver_momentum_20d
    - universe_momentum_20d
)

semiconductor_relative_strength = (
    semiconductor_momentum_20d
    - universe_momentum_20d
)

auto_relative_strength = (
    auto_momentum_20d
    - universe_momentum_20d
)

finance_relative_strength = (
    finance_momentum_20d
    - universe_momentum_20d
)

In [14]:
#leadership feature 묶

leadership_features = pd.DataFrame({
    "Leader": leader_ticker,
    "Leader_Momentum_20D": leader_return,
    "Leadership_Gap_20D": leadership_gap,

    "NAVER_Momentum_20D":
        naver_momentum_20d,

    "Semiconductor_Momentum_20D":
        semiconductor_momentum_20d,

    "Auto_Momentum_20D":
        auto_momentum_20d,

    "Finance_Momentum_20D":
        finance_momentum_20d,

    "NAVER_Relative_Strength":
        naver_relative_strength,

    "Semiconductor_Relative_Strength":
        semiconductor_relative_strength,

    "Auto_Relative_Strength":
        auto_relative_strength,

    "Finance_Relative_Strength":
        finance_relative_strength
})

leadership_features.head()

,Leader,Leader_Momentum_20D,Leadership_Gap_20D,NAVER_Momentum_20D,Semiconductor_Momentum_20D,Auto_Momentum_20D,Finance_Momentum_20D,NAVER_Relative_Strength,Semiconductor_Relative_Strength,Auto_Relative_Strength,Finance_Relative_Strength
date,,,,,,,,,,,
2025-04-16,105560.KS,-0.014778,0.044624,-0.113462,-0.106343,-0.109314,-0.014778,-0.023414,-0.016295,-0.019266,0.075270
2025-04-17,105560.KS,-0.024301,0.054978,-0.117308,-0.122973,-0.102716,-0.024301,-0.019253,-0.024919,-0.004662,0.073753
2025-04-18,105560.KS,0.013530,0.100359,-0.105012,-0.143169,-0.086829,0.013530,-0.012082,-0.050239,0.006101,0.106460
2025-04-21,105560.KS,0.012225,0.091082,-0.094203,-0.121935,-0.120657,0.012225,-0.004902,-0.032634,-0.031356,0.101526
2025-04-22,105560.KS,0.018315,0.084821,-0.066506,-0.119613,-0.156364,0.018315,0.022250,-0.030857,-0.067607,0.107071


In [19]:
#기존 regime feature와 합치기

market_state_2025 = pd.concat(
    [
        regime_features,
        leadership_features
    ],
    axis=1
)

market_state_2025 = (
    market_state_2025
    .dropna()
)

market_state_2025.head()

,Breadth_20D,Momentum_20D,Dispersion_20D,Leader,Leader_Momentum_20D,Leadership_Gap_20D,NAVER_Momentum_20D,Semiconductor_Momentum_20D,Auto_Momentum_20D,Finance_Momentum_20D,NAVER_Relative_Strength,Semiconductor_Relative_Strength,Auto_Relative_Strength,Finance_Relative_Strength
date,,,,,,,,,,,,,,
2025-04-16,0.45,-0.088275,0.019067,105560.KS,-0.014778,0.044624,-0.113462,-0.106343,-0.109314,-0.014778,-0.023414,-0.016295,-0.019266,0.075270
2025-04-17,0.45,-0.096330,0.018548,105560.KS,-0.024301,0.054978,-0.117308,-0.122973,-0.102716,-0.024301,-0.019253,-0.024919,-0.004662,0.073753
2025-04-18,0.45,-0.092261,0.018449,105560.KS,0.013530,0.100359,-0.105012,-0.143169,-0.086829,0.013530,-0.012082,-0.050239,0.006101,0.106460
2025-04-21,0.47,-0.088425,0.017408,105560.KS,0.012225,0.091082,-0.094203,-0.121935,-0.120657,0.012225,-0.004902,-0.032634,-0.031356,0.101526
2025-04-22,0.46,-0.088465,0.017421,105560.KS,0.018315,0.084821,-0.066506,-0.119613,-0.156364,0.018315,0.022250,-0.030857,-0.067607,0.107071


In [20]:
display(
    market_state_2025.loc[
        "2025-06-02":"2025-06-30"
    ].tail()
)

display(
    market_state_2025.loc[
        "2025-10-01":"2025-10-31"
    ].tail()
)

,Breadth_20D,Momentum_20D,Dispersion_20D,Leader,Leader_Momentum_20D,Leadership_Gap_20D,NAVER_Momentum_20D,Semiconductor_Momentum_20D,Auto_Momentum_20D,Finance_Momentum_20D,NAVER_Relative_Strength,Semiconductor_Relative_Strength,Auto_Relative_Strength,Finance_Relative_Strength
date,,,,,,,,,,,,,,
2025-06-24,0.65,0.273365,0.022785,035420.KS,0.586565,0.191550,0.586565,0.255625,0.160690,0.134343,0.307995,-0.022944,-0.117880,-0.144226
2025-06-25,0.63,0.261391,0.023998,035420.KS,0.507979,0.096567,0.507979,0.266035,0.209225,0.073529,0.243418,0.001474,-0.055335,-0.191031
2025-06-26,0.65,0.245283,0.025482,000660.KS,0.449527,0.029505,0.420022,0.283205,0.167432,0.078508,0.173547,0.036731,-0.079043,-0.167966
2025-06-27,0.63,0.210670,0.025597,035420.KS,0.371870,0.004019,0.371870,0.231089,0.117775,0.095049,0.162495,0.021715,-0.091599,-0.114325
2025-06-30,0.61,0.195235,0.025787,035420.KS,0.386688,0.009329,0.386688,0.224925,0.065445,0.079844,0.190322,0.028560,-0.130920,-0.116521


,Breadth_20D,Momentum_20D,Dispersion_20D,Leader,Leader_Momentum_20D,Leadership_Gap_20D,NAVER_Momentum_20D,Semiconductor_Momentum_20D,Auto_Momentum_20D,Finance_Momentum_20D,NAVER_Relative_Strength,Semiconductor_Relative_Strength,Auto_Relative_Strength,Finance_Relative_Strength
date,,,,,,,,,,,,,,
2025-10-27,0.57,0.196632,0.024023,000660.KS,0.524217,0.297210,0.081897,0.375612,0.167431,0.009410,-0.120096,0.173620,-0.034561,-0.192582
2025-10-28,0.55,0.172298,0.023899,000660.KS,0.443213,0.263238,0.095238,0.311594,0.143836,0.008666,-0.078948,0.137409,-0.030350,-0.165520
2025-10-29,0.58,0.214014,0.024958,000660.KS,0.560839,0.374632,0.162281,0.371452,0.186207,0.006071,-0.057212,0.151960,-0.033286,-0.213422
2025-10-30,0.59,0.198529,0.023785,000660.KS,0.593268,0.372070,0.005906,0.403861,0.221198,0.013123,-0.203684,0.194271,0.011608,-0.196466
2025-10-31,0.62,0.264464,0.024642,000660.KS,0.661218,0.309237,0.042885,0.478746,0.351981,0.034605,-0.234508,0.201353,0.074589,-0.242788


In [21]:
#월평균

monthly_market_state = (
    market_state_2025
    .drop(columns=["Leader"])
    .resample("ME")
    .mean()
)

monthly_market_state.index = (
    monthly_market_state
    .index
    .strftime("%Y-%m")
)

monthly_market_state.round(3)

,Breadth_20D,Momentum_20D,Dispersion_20D,Leader_Momentum_20D,Leadership_Gap_20D,NAVER_Momentum_20D,Semiconductor_Momentum_20D,Auto_Momentum_20D,Finance_Momentum_20D,NAVER_Relative_Strength,Semiconductor_Relative_Strength,Auto_Relative_Strength,Finance_Relative_Strength
date,,,,,,,,,,,,,
2025-04,0.472,-0.058,0.017,0.048,0.085,-0.049,-0.101,-0.092,0.048,0.010,-0.042,-0.033,0.107
2025-05,0.524,0.073,0.015,0.225,0.101,-0.000,0.064,0.020,0.225,-0.075,-0.011,-0.055,0.151
2025-06,0.572,0.152,0.019,0.315,0.073,0.226,0.165,0.080,0.129,0.073,0.012,-0.073,-0.024
2025-07,0.533,0.086,0.024,0.214,0.067,0.087,0.108,0.054,0.062,0.003,0.024,-0.030,-0.021
2025-08,0.503,-0.012,0.018,0.085,0.071,-0.076,0.013,0.018,-0.034,-0.063,0.027,0.031,-0.021
2025-09,0.594,0.082,0.017,0.221,0.110,0.056,0.153,0.019,0.034,-0.027,0.070,-0.063,-0.049
2025-10,0.573,0.194,0.024,0.496,0.215,0.100,0.387,0.116,0.011,-0.100,0.187,-0.084,-0.189
2025-11,0.530,0.109,0.024,0.305,0.170,0.007,0.167,0.122,0.088,-0.103,0.057,0.012,-0.022
2025-12,0.467,0.027,0.020,0.124,0.044,-0.070,0.044,0.095,0.016,-0.095,0.018,0.069,-0.010


In [22]:
#저장
market_state_2025.to_csv(
    RESULTS_DIR / "market_state_features_2025.csv",
    encoding="utf-8-sig"
)

monthly_market_state.to_csv(
    RESULTS_DIR / "monthly_market_state_2025.csv",
    encoding="utf-8-sig"
)

print("Leadership / Market State 저장 완료")

Leadership / Market State 저장 완료


In [23]:
# 현재 주도 그룹 자동 판정

relative_strength_cols = {
    "NAVER": "NAVER_Relative_Strength",
    "Semiconductor": "Semiconductor_Relative_Strength",
    "Auto": "Auto_Relative_Strength",
    "Finance": "Finance_Relative_Strength"
}

relative_strength_df = (
    market_state_2025[
        list(relative_strength_cols.values())
    ]
    .rename(
        columns={
            v: k
            for k, v in relative_strength_cols.items()
        }
    )
)

market_state_2025["Leadership_Group"] = (
    relative_strength_df.idxmax(axis=1)
)

In [24]:
market_state_2025[
    [
        "Leadership_Group",
        "NAVER_Relative_Strength",
        "Semiconductor_Relative_Strength",
        "Auto_Relative_Strength",
        "Finance_Relative_Strength"
    ]
].head()

,Leadership_Group,NAVER_Relative_Strength,Semiconductor_Relative_Strength,Auto_Relative_Strength,Finance_Relative_Strength
date,,,,,
2025-04-16,Finance,-0.023414,-0.016295,-0.019266,0.075270
2025-04-17,Finance,-0.019253,-0.024919,-0.004662,0.073753
2025-04-18,Finance,-0.012082,-0.050239,0.006101,0.106460
2025-04-21,Finance,-0.004902,-0.032634,-0.031356,0.101526
2025-04-22,Finance,0.022250,-0.030857,-0.067607,0.107071


In [26]:
#상승/하락 regime
#bull:상승, bear:하락

market_state_2025["Market_Direction"] = np.where(
    market_state_2025["Momentum_20D"] > 0,
    "Bull",
    "Bear"
)

In [27]:
#broad: 최근 20일 동안 비교적 많은 종목이 함께 상승
#narrow: 상승이 일부 종목에 상대적으로 집중

BREADTH_THRESHOLD = 0.55

market_state_2025["Breadth_Regime"] = np.where(
    market_state_2025["Breadth_20D"] >= BREADTH_THRESHOLD,
    "Broad",
    "Narrow"
)

In [28]:
#leadership이 강한지도 구분

LEADERSHIP_GAP_THRESHOLD = (
    market_state_2025[
        "Leadership_Gap_20D"
    ]
    .median()
)

print(
    "Leadership Gap threshold:",
    LEADERSHIP_GAP_THRESHOLD
)

Leadership Gap threshold: 0.08576714791245343


In [29]:
market_state_2025["Leadership_Strength"] = np.where(
    market_state_2025[
        "Leadership_Gap_20D"
    ] >= LEADERSHIP_GAP_THRESHOLD,
    "Strong",
    "Weak"
)

In [30]:
#regime label

market_state_2025["Regime_Label"] = (
    market_state_2025["Market_Direction"]
    + "_"
    + market_state_2025["Breadth_Regime"]
    + "_"
    + market_state_2025["Leadership_Group"]
    + "_"
    + market_state_2025["Leadership_Strength"]
)

In [31]:
#확인

market_state_2025[
    [
        "Momentum_20D",
        "Breadth_20D",
        "Leadership_Group",
        "Leadership_Strength",
        "Regime_Label"
    ]
].tail(20)

,Momentum_20D,Breadth_20D,Leadership_Group,Leadership_Strength,Regime_Label
date,,,,,
2025-12-02,-0.015129,0.43,Finance,Strong,Bear_Narrow_Finance_Strong
2025-12-03,-0.004527,0.45,Finance,Weak,Bear_Narrow_Finance_Weak
2025-12-04,0.003638,0.45,Auto,Weak,Bull_Narrow_Auto_Weak
2025-12-05,0.055184,0.50,Auto,Strong,Bull_Narrow_Auto_Strong
2025-12-08,0.036522,0.49,Auto,Strong,Bull_Narrow_Auto_Strong
2025-12-09,0.007003,0.45,Auto,Strong,Bull_Narrow_Auto_Strong
2025-12-10,-0.004331,0.43,Auto,Weak,Bear_Narrow_Auto_Weak
2025-12-11,-0.019097,0.42,Auto,Weak,Bear_Narrow_Auto_Weak
2025-12-12,0.041955,0.47,Auto,Weak,Bull_Narrow_Auto_Weak


In [32]:
#6월과 10월비교

print("=== 2025 June ===")

display(
    market_state_2025.loc[
        "2025-06-01":"2025-06-30",
        [
            "Momentum_20D",
            "Breadth_20D",
            "Leadership_Group",
            "Leadership_Gap_20D",
            "Regime_Label"
        ]
    ].tail()
)

print("=== 2025 October ===")

display(
    market_state_2025.loc[
        "2025-10-01":"2025-10-31",
        [
            "Momentum_20D",
            "Breadth_20D",
            "Leadership_Group",
            "Leadership_Gap_20D",
            "Regime_Label"
        ]
    ].tail()
)

=== 2025 June ===


,Momentum_20D,Breadth_20D,Leadership_Group,Leadership_Gap_20D,Regime_Label
date,,,,,
2025-06-24,0.273365,0.65,NAVER,0.191550,Bull_Broad_NAVER_Strong
2025-06-25,0.261391,0.63,NAVER,0.096567,Bull_Broad_NAVER_Strong
2025-06-26,0.245283,0.65,NAVER,0.029505,Bull_Broad_NAVER_Weak
2025-06-27,0.210670,0.63,NAVER,0.004019,Bull_Broad_NAVER_Weak
2025-06-30,0.195235,0.61,NAVER,0.009329,Bull_Broad_NAVER_Weak


=== 2025 October ===


,Momentum_20D,Breadth_20D,Leadership_Group,Leadership_Gap_20D,Regime_Label
date,,,,,
2025-10-27,0.196632,0.57,Semiconductor,0.297210,Bull_Broad_Semiconductor_Strong
2025-10-28,0.172298,0.55,Semiconductor,0.263238,Bull_Broad_Semiconductor_Strong
2025-10-29,0.214014,0.58,Semiconductor,0.374632,Bull_Broad_Semiconductor_Strong
2025-10-30,0.198529,0.59,Semiconductor,0.372070,Bull_Broad_Semiconductor_Strong
2025-10-31,0.264464,0.62,Semiconductor,0.309237,Bull_Broad_Semiconductor_Strong


In [ ]:
#regime 규칙
#Bear → Cash, Bull + Strong NAVER leadership → EIIE
#Bull + Strong Semiconductor leadership → Static MVO, 그 외 Bull → Equal Weight

In [33]:
MOMENTUM_THRESHOLD = 0.0
BREADTH_THRESHOLD = 0.55
LEADERSHIP_GAP_THRESHOLD = 0.08576714791245343

In [34]:
def select_strategy(row):

    # 하락장
    if row["Momentum_20D"] <= MOMENTUM_THRESHOLD:
        return "Cash"

    # 상승장 + 강한 주도주 존재
    if row["Leadership_Gap_20D"] >= LEADERSHIP_GAP_THRESHOLD:

        if row["Leadership_Group"] == "NAVER":
            return "EIIE 40ep"

        if row["Leadership_Group"] == "Semiconductor":
            return "Static MVO"

    # 그 외에는 중립적인 분산 전략
    return "Equal Weight"

In [35]:
market_state_2025["Selected_Strategy_V1"] = (
    market_state_2025.apply(
        select_strategy,
        axis=1
    )
)

market_state_2025[
    [
        "Momentum_20D",
        "Leadership_Group",
        "Leadership_Gap_20D",
        "Selected_Strategy_V1"
    ]
].tail(20)

,Momentum_20D,Leadership_Group,Leadership_Gap_20D,Selected_Strategy_V1
date,,,,
2025-12-02,-0.015129,Finance,0.104682,Cash
2025-12-03,-0.004527,Finance,0.051821,Cash
2025-12-04,0.003638,Auto,0.004601,Equal Weight
2025-12-05,0.055184,Auto,0.097447,Equal Weight
2025-12-08,0.036522,Auto,0.089148,Equal Weight
2025-12-09,0.007003,Auto,0.104937,Equal Weight
2025-12-10,-0.004331,Auto,0.061076,Cash
2025-12-11,-0.019097,Auto,0.027509,Cash
2025-12-12,0.041955,Auto,0.003269,Equal Weight


In [36]:
strategy_rule_config = pd.DataFrame({
    "Parameter": [
        "MOMENTUM_THRESHOLD",
        "BREADTH_THRESHOLD",
        "LEADERSHIP_GAP_THRESHOLD"
    ],
    "Value": [
        MOMENTUM_THRESHOLD,
        BREADTH_THRESHOLD,
        LEADERSHIP_GAP_THRESHOLD
    ]
})

strategy_rule_config.to_csv(
    RESULTS_DIR / "regime_strategy_v1_config.csv",
    index=False,
    encoding="utf-8-sig"
)

strategy_rule_config

,Parameter,Value
0,MOMENTUM_THRESHOLD,0.000000
1,BREADTH_THRESHOLD,0.550000
2,LEADERSHIP_GAP_THRESHOLD,0.085767


In [37]:
market_state_2025.to_csv(
    RESULTS_DIR / "market_state_2025_with_strategy_v1.csv",
    encoding="utf-8-sig"
)

In [39]:
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [40]:
KOREA_STOCKS = [
    "005930.KS",
    "000660.KS",
    "005380.KS",
    "035420.KS",
    "105560.KS"
]

TEST_2026_START = "2026-01-01"

# end는 일반적으로 exclusive하게 처리되므로
# 9/15까지 포함하기 위해 다음 날짜 지정
TEST_2026_END = "2026-09-16"

In [41]:
raw_2026 = YahooDownloader(
    start_date=TEST_2026_START,
    end_date=TEST_2026_END,
    ticker_list=KOREA_STOCKS
).fetch_data()

raw_2026["date"] = pd.to_datetime(
    raw_2026["date"]
)

print(raw_2026.shape)

raw_2026.head()

YF deprecation warning: set proxy via new config function: yf.set_config(proxy=proxy)


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Shape of DataFrame:  (865, 8)
(865, 8)


Price,date,close,high,low,open,volume,tic,day
0,2026-01-02,675493.625000,677489.174852,645560.377216,649551.476920,4181895,000660.KS,4
1,2026-01-02,294244.375000,297694.476549,288822.786851,295230.118300,955205,005380.KS,4
2,2026-01-02,128093.312500,128093.312500,119819.581031,119819.581031,30463279,005930.KS,4
3,2026-01-02,244432.375000,246411.584514,235525.932186,240968.758350,1362199,035420.KS,4
4,2026-01-02,120421.148438,121788.460747,119932.822613,120714.143932,648227,105560.KS,4


In [42]:
#공통 거래일 필터

date_counts_2026 = (
    raw_2026
    .groupby("date")["tic"]
    .nunique()
)

common_dates_2026 = date_counts_2026[
    date_counts_2026 == len(KOREA_STOCKS)
].index

raw_2026 = (
    raw_2026[
        raw_2026["date"].isin(common_dates_2026)
    ]
    .sort_values(["date", "tic"])
    .reset_index(drop=True)
)

print(
    raw_2026.groupby("tic").size()
)

tic
000660.KS    173
005380.KS    173
005930.KS    173
035420.KS    173
105560.KS    173
dtype: int64


In [43]:
#가격표

prices_2026 = (
    raw_2026
    .pivot(
        index="date",
        columns="tic",
        values="close"
    )
    .sort_index()
)

print(prices_2026.shape)
print(prices_2026.index.min())
print(prices_2026.index.max())

prices_2026.head()

(173, 5)
2026-01-02 00:00:00
2026-09-15 00:00:00


tic,000660.KS,005380.KS,005930.KS,035420.KS,105560.KS
date,,,,,
2026-01-02,675493.6250,294244.37500,128093.312500,244432.375000,120421.148438
2026-01-05,694451.3750,300158.87500,137662.937500,246906.390625,123839.429688
2026-01-06,724384.6250,303608.93750,138460.390625,257297.234375,123448.765625
2026-01-07,740348.9375,345503.03125,140553.765625,249875.203125,121788.468750
2026-01-08,754317.8750,335645.62500,138360.703125,248390.796875,120616.484375


In [44]:
#2025-2026 가격 연결

stock_prices_2025 = pd.read_csv(
    RESULTS_DIR / "stock_prices_2025.csv",
    index_col=0,
    parse_dates=True
)

prices_for_2026_features = pd.concat(
    [
        stock_prices_2025,
        prices_2026
    ]
)

prices_for_2026_features = (
    prices_for_2026_features
    [~prices_for_2026_features.index.duplicated(keep="last")]
    .sort_index()
)

print(prices_for_2026_features.index.min())
print(prices_for_2026_features.index.max())
print(prices_for_2026_features.shape)

2025-03-19 00:00:00
2026-09-15 00:00:00
(365, 5)


In [45]:
#2026 일간수익률

stock_daily_returns_all = (
    prices_for_2026_features
    .pct_change()
)

In [46]:
#breadth

market_breadth_all = (
    (stock_daily_returns_all > 0)
    .sum(axis=1)
    / len(KOREA_STOCKS)
)

market_breadth_all.loc[
    stock_daily_returns_all.isna().all(axis=1)
] = np.nan

breadth_20d_all = (
    market_breadth_all
    .rolling(20)
    .mean()
)

In [47]:
#universe momentum

universe_daily_return_all = (
    stock_daily_returns_all
    .mean(axis=1)
)

momentum_20d_all = (
    (1 + universe_daily_return_all)
    .rolling(20)
    .apply(np.prod, raw=True)
    - 1
)

In [48]:
#dispersion

dispersion_20d_all = (
    stock_daily_returns_all
    .std(axis=1)
    .rolling(20)
    .mean()
)

In [49]:
#종목별 20일 momentum

stock_momentum_20d_all = (
    prices_for_2026_features
    / prices_for_2026_features.shift(20)
    - 1
)

valid_stock_momentum_20d_all = (
    stock_momentum_20d_all
    .dropna(how="all")
)

In [50]:
#leadership

leader_ticker_all = (
    valid_stock_momentum_20d_all
    .idxmax(axis=1)
)

leader_return_all = (
    valid_stock_momentum_20d_all
    .max(axis=1)
)

In [51]:
#leadership gap

sorted_momentum_all = np.sort(
    valid_stock_momentum_20d_all.values,
    axis=1
)

leadership_gap_all = pd.Series(
    sorted_momentum_all[:, -1]
    - sorted_momentum_all[:, -2],
    index=valid_stock_momentum_20d_all.index,
    name="Leadership_Gap_20D"
)

In [52]:
#그룹별 momentum

semiconductor_momentum_20d_all = (
    valid_stock_momentum_20d_all[
        ["000660.KS", "005930.KS"]
    ]
    .mean(axis=1)
)

naver_momentum_20d_all = (
    valid_stock_momentum_20d_all[
        "035420.KS"
    ]
)

auto_momentum_20d_all = (
    valid_stock_momentum_20d_all[
        "005380.KS"
    ]
)

finance_momentum_20d_all = (
    valid_stock_momentum_20d_all[
        "105560.KS"
    ]
)

universe_momentum_stock_all = (
    valid_stock_momentum_20d_all
    .mean(axis=1)
)

In [ ]:
#momentum_20d_all → 매일 5종목 평균수익률을 만든 뒤 20일 복리

#universe_momentum_stock_all → 각 종목의 20일 수익률을 먼저 만든 뒤 5종목 평균

In [53]:
#relative strength

naver_relative_strength_all = (
    naver_momentum_20d_all
    - universe_momentum_stock_all
)

semiconductor_relative_strength_all = (
    semiconductor_momentum_20d_all
    - universe_momentum_stock_all
)

auto_relative_strength_all = (
    auto_momentum_20d_all
    - universe_momentum_stock_all
)

finance_relative_strength_all = (
    finance_momentum_20d_all
    - universe_momentum_stock_all
)

In [54]:
#하나로 묶기

market_state_all = pd.DataFrame({
    "Breadth_20D":
        breadth_20d_all,

    "Momentum_20D":
        momentum_20d_all,

    "Dispersion_20D":
        dispersion_20d_all,

    "Leader":
        leader_ticker_all,

    "Leader_Momentum_20D":
        leader_return_all,

    "Leadership_Gap_20D":
        leadership_gap_all,

    "NAVER_Relative_Strength":
        naver_relative_strength_all,

    "Semiconductor_Relative_Strength":
        semiconductor_relative_strength_all,

    "Auto_Relative_Strength":
        auto_relative_strength_all,

    "Finance_Relative_Strength":
        finance_relative_strength_all
})

market_state_all = (
    market_state_all
    .dropna()
)

In [ ]:
#2026만 자름

